In [32]:
!pip install -qU langchain langchain-groq langchain-core langgraph

In [33]:
import os
from google.colab import userdata

# Puxa a chave dos "Secrets" do Colab para o ambiente
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Chave da Groq carregada com sucesso!")

Chave da Groq carregada com sucesso!


In [34]:
import json
import unicodedata
from datetime import datetime, timedelta, time
from typing import Dict, List, Any

# Importações do LangGraph e Groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

def normalize(text: str) -> str:
    text = text.lower().strip()
    return "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")

def fmt_time(dt: datetime) -> str:
    return dt.strftime("%Hh%M")

# --- MOCK DE DADOS DA SPRINT 1 ---
def build_mock_data() -> Dict[str, Any]:
    now = datetime.now()
    today = now.date()

    chargers = [
        {"id": "WC-SP-01", "site": "shopping paulista", "power_kw": 60, "status": "charging"},
        {"id": "WC-SP-02", "site": "shopping paulista", "power_kw": 60, "status": "charging"},
        {"id": "WC-SP-03", "site": "shopping paulista", "power_kw": 22, "status": "available"},
        {"id": "WC-SP-04", "site": "shopping paulista", "power_kw": 22, "status": "available"},
        {"id": "WC-SP-05", "site": "shopping paulista", "power_kw": 30, "status": "ready"},
        {"id": "WC-GRL-01", "site": "graal dutra", "power_kw": 60, "status": "available"},
        {"id": "WC-GRL-02", "site": "graal dutra", "power_kw": 60, "status": "fault", "fault_type": "erro de comunicação", "fault_since": datetime.combine(today, time(14, 32))},
    ]

    sessions = []
    today_sessions = [("WC-SP-01", 210.0), ("WC-SP-02", 180.0), ("WC-GRL-01", 250.0), ("WC-SP-05", 260.0), ("WC-SP-04", 140.0), ("WC-GRL-03", 200.0)]
    for i, (cid, kwh) in enumerate(today_sessions):
        sessions.append({"start": datetime.combine(today, time(8+i, 0))})

    return {"generated_at": now, "chargers": chargers, "sessions": sessions}

class WeChargeDataRouter:
    """Classe responsável por rotear e separar dados sensíveis de Operação (B2B) e Motorista (B2C)"""
    def __init__(self, db: Dict[str, Any]):
        self.db = db

    def route_query(self, question: str) -> Dict[str, str]:
        q = normalize(question)
        site_key = "shopping paulista" if "paulista" in q else "graal dutra" if "graal" in q else None

        # Identifica se a pergunta é de Operação (B2B) ou de um Motorista (B2C)
        is_operator = any(word in q for word in ["receita", "faturamento", "energia", "kwh entregue", "rede", "falha", "operacao", "operador"])
        profile = "OPERADOR (Acesso a finanças e infraestrutura)" if is_operator else "MOTORISTA (Acesso apenas à sua experiência, reserva e rotas)"

        facts = ""

        # Lógica B2B - Operacional e Financeiro
        if is_operator:
            if "operacao" in q and site_key:
                ch = [c for c in self.db["chargers"] if c["site"] == site_key]
                op = [c["id"] for c in ch if c["status"] in ["charging", "ready"]]
                av = [c["id"] for c in ch if c["status"] == "available"]
                facts = f"Dados Operacionais: Site {site_key}. Total carregadores: {len(ch)}. Em operação: {', '.join(op)}. Disponíveis: {', '.join(av)}."
            elif "energia" in q and "hoje" in q:
                facts = "Kpis Diários: Hoje a rede entregou 1240 kWh (60% solar, 20% bateria, 20% rede). O modo PV Priority está ativo."
            elif "receita" in q and "semana" in q and site_key:
                facts = "Faturamento: Na última semana a receita estimada no Graal da Dutra foi de exatos R$ 12.430,00."
            elif "falha" in q or "comunicacao" in q:
                fs = [f"{c['id']} ({c['fault_type']} desde as {fmt_time(c['fault_since'])})" for c in self.db["chargers"] if c["status"] == "fault"]
                facts = f"Falhas na Rede: Há {len(fs)} carregadores down: {', '.join(fs)}. Recomende reiniciar o gateway local do GRL-02 e abrir chamado."
            else:
                facts = "Instrução Operacional: Você tem acesso aos dados de receita, energia diária e falhas. Como posso ajudar?"

        # Lógica B2C - Motorista (Protegida)
        else:
            if "reserv" in q and "paulista" in q and ("amanha" in q or "amanhã" in question.lower()):
                facts = "Serviço ao Motorista: Temos 2 carregadores de 60 kW livres no Shopping Paulista para amanhã às 19h. Peça o modelo do carro e quanto deseja carregar."
            elif "carregar ate" in q or ("bateria" in q and "ate" in q):
                facts = "Serviço ao Motorista: Bateria de 60 kWh de 20% a 80% requer 36 kWh. No Graal (60 kW) leva cerca de 35 a 45 mins e custa R$ 79,20 (R$ 2,20/kWh)."
            elif "rio" in q and "50%" in q:
                facts = "Serviço ao Motorista: Rota SP-Rio com 50%. Pare no Graal da Dutra (180 km). Tem carregador 60 kW, tarifa de R$ 2,10/kWh."
            elif "pv priority" in q:
                facts = "Serviço ao Motorista: Explicar que a rede usa energia solar primeiro para carregar o carro, uma opção ecológica."
            else:
                facts = "Instrução Motorista: Você pode ajudar a reservar estações, planejar rotas ou estimar custos de carregamento."

        return {"profile": profile, "facts": facts}

# --- CHATBOT (MULTI-PERFIL / LLAMA 3.3) ---
class WeChargeChatbot:
    def __init__(self):
        self.router = WeChargeDataRouter(build_mock_data())

        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.0
        )

        self.system_prompt_base = """Você é o WeCharge (Camada ChargeGrid Intelligence).
Você lida com dois tipos de pessoas: Motoristas e Operadores. Você vai receber abaixo quem está falando com você.

REGRAS:
1. Se estiver falando com MOTORISTA: Seja prestativo, simples, foque na experiência dele (viagem, reserva, custos DELE). Nunca mostre o faturamento da rede ou detalhes técnicos de infraestrutura.
2. Se estiver falando com OPERADOR: Seja técnico, direto e comercial. Mostre lucros, falhas e KPIs livremente.
3. NÃO ALUCINE: Use EXCLUSIVAMENTE os dados em [CONTEXTO FORNECIDO PELA API MOCK]. Nunca invente km ou dinheiro."""

        workflow = StateGraph(MessagesState)

        def call_model(state: MessagesState):
            last_message = state["messages"][-1].content

            # Chama o Router para pegar fatos + decidir perfil do usuário
            route_data = self.router.route_query(last_message)
            profile = route_data["profile"]
            facts = route_data["facts"]

            # Reconstrói a prompt com as barreiras B2B / B2C
            system_message = SystemMessage(
                content=f"{self.system_prompt_base}\n\n[PERFIL ATUAL DETECTADO]: {profile}\n\n[CONTEXTO FORNECIDO PELA API MOCK]\n{facts}"
            )

            messages_to_send = [system_message] + state["messages"]
            response = self.llm.invoke(messages_to_send)
            return {"messages": [response]}

        workflow.add_node("chatbot", call_model)
        workflow.add_edge(START, "chatbot")
        workflow.add_edge("chatbot", END)

        self.memory = MemorySaver()
        self.graph = workflow.compile(checkpointer=self.memory)

    def ask(self, question: str, thread_id: str = "default_session") -> str:
        config = {"configurable": {"thread_id": thread_id}}
        inputs = {"messages": [HumanMessage(content=question)]}
        output = self.graph.invoke(inputs, config)
        return output["messages"][-1].content

bot = WeChargeChatbot()
print("Chatbot WeCharge inicializado! (Com roteamento B2B vs B2C).")

Chatbot WeCharge inicializado! (Com roteamento B2B vs B2C).


In [35]:
from IPython.display import display, Markdown

TEST_CASES = [
    {"q": "Quais carregadores estão em operação agora no eletroposto do Shopping Paulista?"},
    {"q": "Quanto de energia foi entregue hoje em todos os eletropostos da rede WeCharge?"},
    {"q": "Qual foi a receita estimada na última semana no eletroposto WeCharge do Rodoshop Graal da Dutra?"},
    {"q": "Existe algum carregador com falha ou erro de comunicação na rede WeCharge hoje?"},
    {"q": "Como explico para o cliente o modo 'PV Priority' no eletroposto WeCharge?"},
    {"q": "Quero reservar um carregador rápido no WeCharge do Shopping Paulista amanhã às 19h. Dá?"},
    {"q": "Estou com 20% de bateria em um carro com bateria de 60 kWh e quero carregar até 80% no WeCharge do Graal. Quanto tempo e quanto vou pagar mais ou menos?"},
    {"q": "Saindo de São Paulo para o Rio com 50% de bateria, qual eletroposto WeCharge você recomenda para eu parar primeiro?"},
]

print("Executando bateria de testes WeCharge... \n")
md_content = "# Avaliação Chatbot WeCharge\n\n"

for i, test in enumerate(TEST_CASES, 1):
    resposta = bot.ask(test["q"], thread_id=f"test_sess_{i}")
    md_content += f"### Teste {i}\n**Pergunta:** {test['q']}\n\n**WeCharge:** {resposta}\n\n---\n"

with open("sprint2_resultados.md", "w", encoding="utf-8") as f:
    f.write(md_content)

display(Markdown(md_content))
print("\nArquivo 'sprint2_resultados.md' salvo com sucesso!")

Executando bateria de testes WeCharge... 



# Avaliação Chatbot WeCharge

### Teste 1
**Pergunta:** Quais carregadores estão em operação agora no eletroposto do Shopping Paulista?

**WeCharge:** No eletroposto do Shopping Paulista, atualmente estão em operação os carregadores WC-SP-01, WC-SP-02 e WC-SP-05. Isso significa que 3 dos 5 carregadores totais estão ativos e prontos para uso. Os carregadores WC-SP-03 e WC-SP-04 estão disponíveis, mas não estão em operação no momento.

---
### Teste 2
**Pergunta:** Quanto de energia foi entregue hoje em todos os eletropostos da rede WeCharge?

**WeCharge:** De acordo com os KPIs diários, a rede WeCharge entregou um total de 1240 kWh de energia hoje. Essa energia foi gerada por diferentes fontes, com 60% vindo de fontes solares, 20% de baterias e 20% da rede elétrica convencional. Com o modo PV Priority ativo, priorizamos a utilização de energia solar sempre que possível.

---
### Teste 3
**Pergunta:** Qual foi a receita estimada na última semana no eletroposto WeCharge do Rodoshop Graal da Dutra?

**WeCharge:** A receita estimada na última semana no eletroposto WeCharge do Rodoshop Graal da Dutra foi de exatos R$ 12.430,00.

---
### Teste 4
**Pergunta:** Existe algum carregador com falha ou erro de comunicação na rede WeCharge hoje?

**WeCharge:** Sim, há um carregador com falha na rede WeCharge. O carregador WC-GRL-02 está apresentando um erro de comunicação desde as 14h32. Recomendo que você reinicie o gateway local do GRL-02 e abra um chamado para resolver o problema o mais rápido possível. Isso garantirá que o carregador volte a funcionar normalmente e não afete a experiência dos motoristas que utilizam a nossa rede.

---
### Teste 5
**Pergunta:** Como explico para o cliente o modo 'PV Priority' no eletroposto WeCharge?

**WeCharge:** Olá! Estou aqui para ajudar. O modo 'PV Priority' no eletroposto WeCharge é uma opção ecológica incrível que prioriza a utilização de energia solar para carregar o seu carro. Isso significa que, sempre que possível, o nosso sistema irá usar a energia gerada pelos painéis solares para carregar o seu veículo elétrico.

Essa abordagem não apenas reduz a dependência de fontes de energia não renováveis, mas também ajuda a minimizar o impacto ambiental da sua viagem. Além disso, é uma ótima maneira de contribuir para um futuro mais sustentável.

Quando você conectar o seu carro ao eletroposto WeCharge, o sistema automaticamente irá verificar se há energia solar disponível para carregar o seu veículo. Se houver, ele irá priorizar a utilização dessa energia limpa e renovável. É uma forma simples e eficaz de fazer uma escolha ecológica enquanto você está na estrada.

Lembre-se de que, mesmo que não haja energia solar disponível no momento, o nosso sistema irá garantir que o seu carro seja carregado de forma eficiente e segura. A prioridade é sempre a sua conveniência e a experiência de carregamento do seu veículo.

Espero que isso tenha ajudado a esclarecer como funciona o modo 'PV Priority' no eletroposto WeCharge! Se tiver mais alguma dúvida, sinta-se à vontade para perguntar.

---
### Teste 6
**Pergunta:** Quero reservar um carregador rápido no WeCharge do Shopping Paulista amanhã às 19h. Dá?

**WeCharge:** Sim, é possível! Temos 2 carregadores de 60 kW livres no Shopping Paulista para amanhã às 19h. Para garantir a reserva, preciso saber mais sobre o seu veículo. Pode me informar o modelo do seu carro, por favor? Além disso, você gostaria de carregar o veículo até um determinado nível de carga ou por um período específico? Isso ajudará a calcular o tempo e o custo da carga.

---
### Teste 7
**Pergunta:** Estou com 20% de bateria em um carro com bateria de 60 kWh e quero carregar até 80% no WeCharge do Graal. Quanto tempo e quanto vou pagar mais ou menos?

**WeCharge:** Olá! Estou aqui para ajudar.

Para carregar sua bateria de 60 kWh de 20% a 80%, você precisará de 36 kWh. No nosso ponto de carregamento do Graal, que tem uma potência de 60 kW, o tempo de carregamento será de aproximadamente 35 a 45 minutos.

Quanto ao custo, o preço do carregamento é de R$ 2,20 por kWh. Então, para 36 kWh, você pagará cerca de R$ 79,20.

Então, para resumir: você precisará de cerca de 35 a 45 minutos para carregar sua bateria e o custo será de aproximadamente R$ 79,20.

---
### Teste 8
**Pergunta:** Saindo de São Paulo para o Rio com 50% de bateria, qual eletroposto WeCharge você recomenda para eu parar primeiro?

**WeCharge:** Olá! Estou aqui para ajudar.

Considerando que você está saindo de São Paulo para o Rio com 50% de bateria, eu recomendo parar no eletroposto WeCharge mais próximo da sua rota. No entanto, como não tenho informações sobre a localização exata dos eletropostos, posso sugerir que você use nosso mapa de rotas para encontrar o eletroposto mais conveniente.

Mas, posso dizer que, se você precisar carregar sua bateria de 20% a 80%, você precisará de cerca de 36 kWh. Com base nas nossas taxas, isso custaria aproximadamente R$ 79,20, considerando o preço de R$ 2,20/kWh.

Se você quiser, posso ajudá-lo a encontrar um eletroposto WeCharge ao longo da sua rota e fornecer mais informações sobre o tempo de carregamento e o custo. Qual é o seu modelo de veículo e qual é a distância que você precisa percorrer?

---



Arquivo 'sprint2_resultados.md' salvo com sucesso!


In [ ]:
print("=" * 60)
print("🔌 Bem-vindo ao WeCharge - ChargeGrid Intelligence")
print("⚡ Operação de Eletropostos GoodWe")
print("=" * 60)
print("Instrução: Digite 'sair' para encerrar a conversa.\n")

thread_id = "video_demo_sessao_1"

while True:
    try:
        user_input = input("Digite aqui sua pergunta: ").strip()

        if user_input.lower() in ['sair', 'exit', 'quit']:
            print("\nWeCharge: Sessão encerrada. Até logo!")
            break

        if not user_input:
            continue

        print("WeCharge está processando...")
        resposta = bot.ask(user_input, thread_id=thread_id)

        print(f"\nWeCharge: {resposta}\n")
        print("-" * 60)

    except KeyboardInterrupt:
        print("\nSessão interrompida pelo usuário.")
        break

🔌 Bem-vindo ao WeCharge - ChargeGrid Intelligence
⚡ Operação de Eletropostos GoodWe
Instrução: Digite 'sair' para encerrar a conversa.

Digite aqui sua pergunta: eu tenho 20% de bateria e quero carregar ate 50%  no carregador do graal dutra, quanto fica
WeCharge está processando...

WeCharge: Olá! Se você quer carregar sua bateria de 20% para 50%, precisamos calcular a quantidade de energia necessária.

Considerando que a bateria tem 60 kWh e você quer carregar de 20% para 50%, vamos calcular a diferença:

50% de 60 kWh = 0,5 x 60 = 30 kWh
20% de 60 kWh = 0,2 x 60 = 12 kWh

Então, você precisa carregar 30 kWh - 12 kWh = 18 kWh.

No carregador do Graal, o custo é de R$ 2,20 por kWh. Portanto, o custo total para carregar 18 kWh seria:

18 kWh x R$ 2,20/kWh = R$ 39,60

Além disso, o tempo de carregamento no Graal (60 kW) seria aproximadamente:

18 kWh / 60 kW = 0,3 horas (ou cerca de 18 minutos)

Então, para carregar sua bateria de 20% para 50% no carregador do Graal, você pagaria aproxim